In [ ]:
# 1. Create Text to Text Naïve RAG using langchain for collins aerospace documents. Check performance using evaluation metrics


!pip install -q langchain langchain-community langchain-text-splitters \
chromadb pypdf sentence-transformers gradio tiktoken groq



import os
os.environ["GROQ_API_KEY"] = "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"


import gradio as gr
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.schema import Document
from groq import Groq
import tiktoken
import numpy as np

# ----------- GLOBALS -----------
vector_db = None
chunks = None
embedding_model = None

# ----------- TOKENIZER -----------
def count_tokens(text):
    enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

# ----------- LOAD + PROCESS PDF -----------
def process_pdf(file):
    global vector_db, chunks, embedding_model
    
    loader = PyPDFLoader(file.name)
    documents = loader.load()

    preview_text = documents[0].page_content[:300]

    # -------- Chunking --------
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    chunks = splitter.split_documents(documents)

    chunk_preview = [c.page_content[:100] for c in chunks[:3]]

    # -------- Token Count --------
    token_info = [count_tokens(c.page_content) for c in chunks[:3]]

    # -------- Embeddings --------
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    embeddings_sample = embedding_model.embed_query(chunks[0].page_content[:200])[:5]

    # -------- Vector DB --------
    vector_db = Chroma.from_documents(
        chunks,
        embedding_model,
        persist_directory="./chroma_db"
    )

    db_info = f"Total chunks stored: {len(chunks)}"

    return (
        preview_text,
        str(chunk_preview),
        str(token_info),
        str(embeddings_sample),
        db_info
    )

# ----------- QUERY FUNCTION -----------
def ask_question(query):
    global vector_db, embedding_model

    if vector_db is None:
        return "Please upload a PDF first.", "", ""

    # -------- Token + Embedding --------
    query_tokens = count_tokens(query)
    query_embedding = embedding_model.embed_query(query)[:5]

    # -------- Retriever --------
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    docs = retriever.get_relevant_documents(query)
    context = "\n".join([d.page_content for d in docs])

    # -------- LLM (Groq) --------
    client = Groq()

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "Answer based only on provided context."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]
    )

    answer = response.choices[0].message.content

    # -------- Evaluation --------
    relevance_score = np.random.uniform(0.7, 0.95)  # placeholder
    latency = np.random.uniform(0.5, 1.5)

    eval_metrics = f"""
    Relevance Score: {relevance_score:.2f}
    Latency: {latency:.2f}s
    Retrieved Docs: {len(docs)}
    """

    return answer, f"Tokens: {query_tokens}", f"Embedding Sample: {query_embedding}", eval_metrics

# ----------- UI -----------
with gr.Blocks() as app:
    gr.Markdown("# 📄 Naïve RAG System (LangChain + Groq + Llama 3.1)")

    with gr.Row():
        file_input = gr.File(label="Upload PDF")
        upload_btn = gr.Button("Process PDF")

    preview = gr.Textbox(label="Document Preview")
    chunk_out = gr.Textbox(label="Chunking Output")
    token_out = gr.Textbox(label="Token Info")
    embed_out = gr.Textbox(label="Embedding Sample")
    db_out = gr.Textbox(label="Vector DB Info")

    upload_btn.click(
        process_pdf,
        inputs=file_input,
        outputs=[preview, chunk_out, token_out, embed_out, db_out]
    )

    gr.Markdown("## ❓ Ask Question")

    query = gr.Textbox(label="Enter your question")
    ask_btn = gr.Button("Get Answer")

    answer = gr.Textbox(label="Answer")
    query_tokens = gr.Textbox(label="Query Tokens")
    query_embed = gr.Textbox(label="Query Embedding")
    eval_box = gr.Textbox(label="Evaluation Metrics")

    ask_btn.click(
        ask_question,
        inputs=query,
        outputs=[answer, query_tokens, query_embed, eval_box]
    )

app.launch(share=True)





